# 🔀 EnerGIS Workflow Comparison Dashboard

Vergleiche mehrere Workflow-Simulationen nebeneinander.

## Features

- 📊 **Kosten-Vergleich**: Gesamtkosten und Breakdown mehrerer Workflows
- 🏭 **Design-Vergleich**: Anlagen-Kapazitäten Side-by-Side
- 📈 **Performance-Metriken**: Optimality Gaps und KPIs
- 🔬 **Sensitivitäts-Analyse**: Parametervariation visualisieren

## Verwendung

1. Führe Setup-Zellen aus
2. Wähle 2+ Workflows zum Vergleichen
3. Analysiere Unterschiede in interaktiven Plots

---

## 1. Setup

In [ ]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# Auto-Setup mit notebook_helpers
from energis.io.notebook_helpers import setup_notebook_environment

PROJECT_ROOT = setup_notebook_environment()
print("\n✅ Setup abgeschlossen")

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

from energis.io.notebook_helpers import (
    list_saved_workflows,
    load_workflow_from_saved
)

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAVE_PLOTLY = True
    print("✅ Plotly verfügbar")
except ImportError:
    HAVE_PLOTLY = False
    print("⚠️  Plotly nicht verfügbar - installiere mit: pip install plotly")

print("✅ Imports erfolgreich")

## 2. Workflows auswählen

In [ ]:
# Liste alle verfügbaren Workflows
workflows = list_saved_workflows(sort_by="date")

if len(workflows) < 2:
    print("⚠️  Weniger als 2 Workflows gefunden!")
    print(f"   Verfügbar: {len(workflows)}")
    print("\n💡 Erstelle mehr Workflows:")
    print("   - Führe runner.ipynb oder scenario_studio.ipynb mehrfach aus")
    print("   - Variiere Parameter zwischen Runs")
else:
    print(f"📦 {len(workflows)} Workflows gefunden:\n")
    print(f"{'#':<4} {'Name':<40} {'Kosten [EUR]':<15} {'Steps'}")
    print("-" * 80)
    
    for i, wf in enumerate(workflows, 1):
        name = wf['name'][:38] + ".." if len(wf['name']) > 40 else wf['name']
        costs = f"{wf['costs']:,.0f}" if wf['costs'] > 0 else 'N/A'
        steps = ' → '.join(wf['steps']) if wf['steps'] else 'N/A'
        print(f"{i:<4} {name:<40} {costs:<15} {steps}")

In [ ]:
# Wähle Workflows zum Vergleichen (manuelle Auswahl)
# Option 1: Nach Index auswählen
SELECTED_INDICES = [1, 2]  # Ändere diese Indices nach Bedarf

# Option 2: Nach Namen auswählen (kommentiere aus wenn gewünscht)
# SELECTED_NAMES = ["Baseline Simulation", "Optimized"]

# Lade ausgewählte Workflows
if 'SELECTED_NAMES' in locals():
    selected_workflows = [
        wf for wf in workflows 
        if wf['name'] in SELECTED_NAMES
    ]
else:
    selected_workflows = [
        workflows[i-1] for i in SELECTED_INDICES 
        if 0 < i <= len(workflows)
    ]

if len(selected_workflows) < 2:
    print(f"⚠️  Nur {len(selected_workflows)} Workflow(s) ausgewählt")
    print("   Mindestens 2 benötigt für Vergleich")
    comparison_possible = False
else:
    print(f"✅ {len(selected_workflows)} Workflows zum Vergleichen ausgewählt:\n")
    for wf in selected_workflows:
        print(f"  • {wf['name']}")
        print(f"    Kosten: {wf['costs']:,.0f} EUR")
        print(f"    Steps: {' → '.join(wf['steps'])}")
        print()
    comparison_possible = True

## 3. Workflows laden

In [ ]:
if comparison_possible:
    # Lade alle ausgewählten Workflows
    loaded_workflows = []
    
    print("📂 Lade Workflows...\n")
    
    for wf_meta in selected_workflows:
        print(f"  Lade: {wf_meta['name']}")
        workflow = load_workflow_from_saved(wf_meta['path'], verbose=False)
        loaded_workflows.append({
            'name': wf_meta['name'],
            'meta': wf_meta,
            'workflow': workflow
        })
    
    print(f"\n✅ Alle {len(loaded_workflows)} Workflows geladen!")
else:
    print("⏭️  Überspringen (nicht genug Workflows)")
    loaded_workflows = []

## 4. Kosten-Vergleich

In [ ]:
if comparison_possible and HAVE_PLOTLY:
    # Sammle Kosten-Daten
    cost_data = []
    
    for wf in loaded_workflows:
        result = wf['workflow'].rh_result or wf['workflow'].mpc_result or wf['workflow'].pf_result
        
        if result and result.costs:
            total = result.costs.get('objective.OBJ_value_EUR', 0.0)
            capex = result.costs.get('objective.Capex_cost_EUR', 0.0)
            elec = result.costs.get('objective.Grid_energy_cost_EUR', 0.0)
            fuel = result.costs.get('objective.Fuel_cost_EUR', 0.0)
            
            cost_data.append({
                'name': wf['name'],
                'total': total,
                'capex': capex,
                'electricity': elec,
                'fuel': fuel,
                'opex': total - capex
            })
    
    # Erstelle Vergleichs-Plot
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Gesamtkosten-Vergleich', 'Kosten-Breakdown'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}]]
    )
    
    # Plot 1: Gesamtkosten
    names = [d['name'] for d in cost_data]
    totals = [d['total'] for d in cost_data]
    
    fig.add_trace(
        go.Bar(
            x=names,
            y=totals,
            text=[f"{t:,.0f} €" for t in totals],
            textposition='auto',
            marker=dict(color='#4477AA'),
            name='Gesamtkosten'
        ),
        row=1, col=1
    )
    
    # Plot 2: Breakdown (Stacked Bar)
    fig.add_trace(
        go.Bar(
            x=names,
            y=[d['capex'] for d in cost_data],
            name='CAPEX',
            marker=dict(color='#228833')
        ),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Bar(
            x=names,
            y=[d['electricity'] for d in cost_data],
            name='Strom',
            marker=dict(color='#CCBB44')
        ),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Bar(
            x=names,
            y=[d['fuel'] for d in cost_data],
            name='Brennstoff',
            marker=dict(color='#EE6677')
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        height=500,
        barmode='stack',
        showlegend=True,
        title_text="📊 Kosten-Vergleich",
        title_font_size=20
    )
    
    fig.update_yaxes(title_text="Kosten [EUR]", row=1, col=1)
    fig.update_yaxes(title_text="Kosten [EUR]", row=1, col=2)
    
    fig.show()
    
    # Textuelle Zusammenfassung
    print("\n💰 Kosten-Vergleich:\n")
    print(f"{'Workflow':<40} {'Gesamt':<15} {'Differenz':<15}")
    print("-" * 70)
    
    baseline = cost_data[0]['total']
    for d in cost_data:
        diff = d['total'] - baseline
        diff_pct = (diff / baseline * 100) if baseline > 0 else 0
        diff_str = f"{diff:+,.0f} € ({diff_pct:+.1f}%)"
        print(f"{d['name']:<40} {d['total']:>13,.0f} € {diff_str:<15}")
    
elif comparison_possible and not HAVE_PLOTLY:
    print("⚠️  Plotly nicht verfügbar - Kosten-Vergleich als Tabelle:\n")
    
    for wf in loaded_workflows:
        result = wf['workflow'].rh_result or wf['workflow'].mpc_result or wf['workflow'].pf_result
        if result and result.costs:
            total = result.costs.get('objective.OBJ_value_EUR', 0.0)
            print(f"{wf['name']}: {total:,.0f} EUR")
else:
    print("⏭️  Kosten-Vergleich übersprungen")

## 5. Design-Vergleich (Kapazitäten)

In [ ]:
if comparison_possible and HAVE_PLOTLY:
    # Sammle Design-Daten
    design_data = {}
    
    for wf in loaded_workflows:
        design = wf['workflow'].design
        
        if design:
            design_data[wf['name']] = {
                'heat_pumps': {},
                'storage': 0.0
            }
            
            # Wärmepumpen
            if design.heat_pumps:
                for hp_id, hp_data in design.heat_pumps.items():
                    capacity = hp_data.get('capacity_mw', 0.0)
                    design_data[wf['name']]['heat_pumps'][hp_id] = capacity
            
            # Speicher
            if design.storage:
                design_data[wf['name']]['storage'] = design.storage.get('capacity_mwh', 0.0)
    
    # Erstelle Vergleichs-Plot
    all_hp_ids = set()
    for data in design_data.values():
        all_hp_ids.update(data['heat_pumps'].keys())
    
    fig = go.Figure()
    
    # Wärmepumpen
    for hp_id in sorted(all_hp_ids):
        capacities = [
            design_data[name]['heat_pumps'].get(hp_id, 0.0)
            for name in design_data.keys()
        ]
        
        fig.add_trace(go.Bar(
            x=list(design_data.keys()),
            y=capacities,
            name=hp_id
        ))
    
    # Speicher (separate Achse)
    storage_capacities = [data['storage'] for data in design_data.values()]
    
    fig.add_trace(go.Bar(
        x=list(design_data.keys()),
        y=storage_capacities,
        name='Speicher (MWh)',
        yaxis='y2',
        marker=dict(color='#AA3377')
    ))
    
    fig.update_layout(
        height=500,
        title_text="🏭 Design-Vergleich: Kapazitäten",
        title_font_size=20,
        barmode='group',
        yaxis=dict(title='Wärmepumpen [MW]'),
        yaxis2=dict(
            title='Speicher [MWh]',
            overlaying='y',
            side='right'
        )
    )
    
    fig.show()
    
    # Textuelle Zusammenfassung
    print("\n🏭 Design-Vergleich:\n")
    for name, data in design_data.items():
        print(f"{name}:")
        print(f"  Wärmepumpen:")
        for hp_id, cap in sorted(data['heat_pumps'].items()):
            print(f"    {hp_id}: {cap:.2f} MW")
        print(f"  Speicher: {data['storage']:.2f} MWh")
        print()
        
else:
    print("⏭️  Design-Vergleich übersprungen")

## 6. Performance-Metriken

In [ ]:
if comparison_possible:
    # Sammle Performance-Metriken
    metrics = []
    
    for wf in loaded_workflows:
        pf_result = wf['workflow'].pf_result
        rh_result = wf['workflow'].rh_result or wf['workflow'].mpc_result
        
        metric = {'name': wf['name']}
        
        if pf_result and rh_result:
            # Optimality Gap
            pf_cost = pf_result.costs.get('objective.OBJ_value_EUR', 0.0)
            rh_cost = rh_result.costs.get('objective.OBJ_value_EUR', 0.0)
            
            if pf_cost > 0:
                gap = (rh_cost - pf_cost) / pf_cost * 100
                metric['optimality_gap'] = gap
                metric['pf_cost'] = pf_cost
                metric['rh_cost'] = rh_cost
        
        # Zeitschritte
        primary_result = rh_result or pf_result
        if primary_result:
            metric['timesteps'] = len(primary_result.table)
        
        metrics.append(metric)
    
    # Zeige Metriken
    print("\n📊 Performance-Metriken:\n")
    print(f"{'Workflow':<40} {'Opt. Gap':<12} {'PF Cost':<15} {'RH Cost'}")
    print("-" * 85)
    
    for m in metrics:
        name = m['name'][:38] if len(m['name']) > 40 else m['name']
        
        if 'optimality_gap' in m:
            gap_str = f"{m['optimality_gap']:>6.2f} %"
            pf_str = f"{m['pf_cost']:>13,.0f} €"
            rh_str = f"{m['rh_cost']:>13,.0f} €"
        else:
            gap_str = "N/A"
            pf_str = "N/A"
            rh_str = "N/A"
        
        print(f"{name:<40} {gap_str:<12} {pf_str:<15} {rh_str}")
    
    # Optimality Gap Plot
    if HAVE_PLOTLY and any('optimality_gap' in m for m in metrics):
        gaps = [m.get('optimality_gap', 0) for m in metrics if 'optimality_gap' in m]
        names_with_gap = [m['name'] for m in metrics if 'optimality_gap' in m]
        
        fig = go.Figure()
        
        fig.add_trace(go.Bar(
            x=names_with_gap,
            y=gaps,
            text=[f"{g:.2f}%" for g in gaps],
            textposition='auto',
            marker=dict(
                color=gaps,
                colorscale='RdYlGn_r',
                showscale=True,
                colorbar=dict(title="Gap %")
            )
        ))
        
        fig.update_layout(
            height=400,
            title_text="📉 Optimality Gap (PF vs RH/MPC)",
            title_font_size=20,
            yaxis_title="Optimality Gap [%]",
            showlegend=False
        )
        
        # Threshold-Linien
        fig.add_hline(y=1.0, line_dash="dash", line_color="green", 
                     annotation_text="Exzellent (<1%)")
        fig.add_hline(y=5.0, line_dash="dash", line_color="orange", 
                     annotation_text="Gut (<5%)")
        fig.add_hline(y=10.0, line_dash="dash", line_color="red", 
                     annotation_text="Akzeptabel (<10%)")
        
        fig.show()
        
else:
    print("⏭️  Performance-Metriken übersprungen")

---

## 💡 Interpretation

### Kosten-Vergleich:
- Vergleiche Gesamtkosten zwischen Szenarien
- Analysiere, welche Kostenblöcke sich am meisten unterscheiden
- Identifiziere kostenoptimale Konfigurationen

### Design-Vergleich:
- Vergleiche Anlagen-Kapazitäten
- Erkenne Trends (z.B. größerer Speicher → niedrigere Kosten)
- Validiere Dimensionierung

### Optimality Gap:
- **<1%**: Exzellent - sehr gute operative Planung
- **<5%**: Gut - akzeptable Qualität
- **<10%**: Akzeptabel - Horizont könnte verlängert werden
- **>10%**: Hoch - Horizont zu kurz oder zu viel Unsicherheit

### Sensitivitäts-Analyse:
- Variiere einen Parameter (z.B. CO2-Preis, Speichergröße)
- Erstelle multiple Workflows mit verschiedenen Parametern
- Analysiere Auswirkung auf Kosten und Design

---

## 🎯 Nächste Schritte

- Erstelle mehr Workflows mit verschiedenen Parametern
- Verwende dieses Notebook für systematische Sensitivitäts-Analysen
- Exportiere Vergleichs-Plots für Berichte/Publikationen

---